# AeroMetric-3D | GPU Backend Server
Run this notebook on a **T4 GPU** runtime to process drone videos in minutes.

1. Go to **Runtime > Change runtime type** and select **T4 GPU**.
2. Enter your ngrok authtoken below (get one free at [ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken)).
3. Run all cells (`Ctrl+F9`).
4. Copy the public ngrok URL printed at the bottom and paste it into your web app.

In [ ]:
# 1. Install Dependencies (Takes ~1 min)
!pip install pycolmap ultralytics torch transformers trimesh laspy pyproj rasterio pyngrok flask flask-cors

In [ ]:
# 2. Start Flask Server and ngrok tunnel
import os
import json
import subprocess
import zipfile
import threading
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from pyngrok import ngrok

# ============== IMPORTANT ==============
NGROK_AUTH_TOKEN = ""  # Paste your token here (e.g. '2aX...Y9Z')
# =======================================

if not NGROK_AUTH_TOKEN:
    print("\n\n❌ ERROR: You must provide an NGROK_AUTH_TOKEN!\n\n")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

app = Flask(__name__)
CORS(app)

JOBS = {}

@app.route('/upload', methods=['POST'])
def upload_job():
    if 'scripts_zip' not in request.files or 'video' not in request.files:
        return jsonify({'error': 'Missing files'}), 400
        
    dataset_name = request.form.get('dataset_name', 'dataset_default')
    workspace_dir = f'/content/{dataset_name}_workspace'
    os.makedirs(workspace_dir, exist_ok=True)
    
    # Extract scripts
    zip_path = os.path.join(workspace_dir, 'scripts.zip')
    request.files['scripts_zip'].save(zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(workspace_dir)
        
    # Save video with clean filename (no spaces or special chars)
    video_file = request.files['video']
    safe_name = "".join(c for c in video_file.filename if c.isalnum() or c in "._-") or "input_video.mp4"
    video_path = os.path.join(workspace_dir, safe_name)
    video_file.save(video_path)
    
    dataset_dir = os.path.join(workspace_dir, dataset_name)
    
    JOBS[dataset_name] = {'status': 'processing'}
    
    def run_pipeline():
        cmd = f'python3 "{workspace_dir}/run_pipeline.py" "{video_path}" "{dataset_dir}"'
        process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        
        log_file = os.path.join(workspace_dir, 'pipeline.log')
        with open(log_file, 'w') as f:
            for line in process.stdout:
                f.write(line)
                f.flush()
                
        process.wait()
        if process.returncode == 0:
            JOBS[dataset_name]['status'] = 'completed'
        else:
            JOBS[dataset_name]['status'] = 'failed'
            try:
                with open(log_file, 'r') as f:
                    JOBS[dataset_name]['error'] = f.read()[-1500:]
            except Exception:
                pass

    # Run in background so request returns instantly
    threading.Thread(target=run_pipeline).start()
    
    return jsonify({'success': True, 'datasetName': dataset_name, 'message': 'Processing started on GPU'})

@app.route('/status/<dataset_name>', methods=['GET'])
def status(dataset_name):
    if dataset_name not in JOBS:
        return jsonify({'error': 'Job not found'}), 404
    job = JOBS[dataset_name]
    resp = {'status': job['status']}
    if 'error' in job:
        resp['error'] = job['error']
    # Return real-time progress update if available
    geo_path = f'/content/{dataset_name}_workspace/{dataset_name}/geospatial_metadata.json'
    if os.path.exists(geo_path):
        try:
            with open(geo_path, 'r') as f:
                meta = json.load(f)
                if 'progress' in meta:
                    resp['progress'] = meta['progress']
        except Exception:
            pass
    return jsonify(resp)

@app.route('/model/<dataset_name>', methods=['GET'])
def get_model(dataset_name):
    glb_path = f'/content/{dataset_name}_workspace/{dataset_name}/metric_reconstruction.glb'
    if os.path.exists(glb_path):
        return send_file(glb_path, mimetype='model/gltf-binary')
    return jsonify({'error': 'Model not ready'}), 404
    
@app.route('/metadata/<dataset_name>', methods=['GET'])
def get_metadata(dataset_name):
    geo_path = f'/content/{dataset_name}_workspace/{dataset_name}/geospatial_metadata.json'
    if os.path.exists(geo_path):
        with open(geo_path, 'r') as f:
            return jsonify(json.load(f))
    return jsonify({'error': 'Metadata not ready'}), 404

# Open ngrok tunnel
public_url = ngrok.connect(5000).public_url
print("\n" + "="*60)
print("🚀 AERO-METRIC 3D GPU API IS LIVE!")
print("Copy this URL into your website configuration:")
print(f"--> {public_url} <--")
print("="*60 + "\n")

app.run(port=5000)
